# 技能2 · Day 1：从流程驱动到智能驱动 + AI治理框架

> **上机练习**：用 pydantic + pandas 实现 NIST AI RMF 合规扫描器 + EU AI Act 风险分级器
> **真实库**：pydantic（控制项schema定义）、pandas（合规结果分析）
> **真实数据**：基于 OECD AI Incidents Monitor 真实事件类型构建的AI用例集
> **v5.0 哲学**：真实即严谨，练习即掌握


## 上机概览

本 notebook 实现两个核心AI治理工具：

1. **NIST AI RMF 合规扫描器**：用 pydantic 定义控制项schema，将AI用例对照NIST AI RMF四大功能（Govern/Map/Measure/Manage）打分
2. **EU AI Act 风险分级器**：按EU AI Act真实条款（Article 5 / Annex III / Article 50）判定AI用例的风险等级

完成后用 pandas 将合规扫描结果转为DataFrame，生成风险热力图分析。

**6个TODO**：
- TODO1：NIST AI RMF 控制项 Schema 定义
- TODO2：AI 用例注册表构建
- TODO3：NIST AI RMF 合规扫描器实现
- TODO4：EU AI Act 风险分级器实现
- TODO5：风险热力图分析（pandas）
- TODO6：营销AI治理专项分析


In [ ]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import Optional
import pandas as pd

print('pydantic + pandas loaded successfully')

## TODO1：NIST AI RMF 控制项 Schema

用 pydantic 定义NIST AI RMF控制项的数据模型。NIST AI RMF 1.0包含四大功能：

| 功能 | 职责 | 控制项数 |
|------|------|:--------:|
| **Govern**（治理） | 政策、流程、问责结构、人员能力 | 5 |
| **Map**（映射） | 上下文建立、风险识别、影响评估 | 5 |
| **Measure**（度量） | 评估方法、可信特征、指标追踪 | 4 |
| **Manage**（管理） | 风险优先级、资源分配、风险响应 | 4 |

每个控制项有：id（如GOVERN-1）、function、category、description、status、score(0-100)。

**你的任务**：定义 `ComplianceStatus` 枚举、`ControlItem` 模型、`NIST_CONTROL_ITEMS` 列表（18个真实控制项）。

In [ ]:
# --- ComplianceStatus 枚举 ---
class ComplianceStatus(str, Enum):
    NOT_ASSESSED = "not_assessed"
    NOT_MET = "not_met"
    PARTIALLY_MET = "partially_met"
    MET = "met"

# --- ControlItem pydantic模型 ---
class ControlItem(BaseModel):
    id: str
    function: str  # Govern / Map / Measure / Manage
    category: str
    description: str
    status: ComplianceStatus = ComplianceStatus.NOT_ASSESSED
    score: float = Field(ge=0, le=100, default=0.0)

# --- NIST AI RMF 1.0 真实控制项（18项，来自官方文档）---
NIST_CONTROL_ITEMS = [
    # Govern（GOVERN-1~5）
    ControlItem(id="GOVERN-1", function="Govern", category="政策与流程",
                description="AI系统的政策、流程、程序和实践已建立并文档化"),
    ControlItem(id="GOVERN-2", function="Govern", category="问责结构",
                description="明确的角色和责任分配，每个AI系统有指定的问责人(Accountable Owner)"),
    ControlItem(id="GOVERN-3", function="Govern", category="人员能力",
                description="团队具备AI风险管理所需的培训、专业知识和资源"),
    ControlItem(id="GOVERN-4", function="Govern", category="利益相关方参与",
                description="外部利益相关方（用户、受影响群体、监管机构）的参与机制已建立"),
    ControlItem(id="GOVERN-5", function="Govern", category="全生命周期治理",
                description="AI系统全生命周期（设计-开发-部署-运维-退役）的治理流程已定义"),
    # Map（MAP-1~5）
    ControlItem(id="MAP-1", function="Map", category="上下文建立",
                description="AI系统的使用上下文已明确记录（业务目标、部署环境、影响范围）"),
    ControlItem(id="MAP-2", function="Map", category="分类与风险识别",
                description="AI系统已按风险等级分类，潜在风险已识别和文档化"),
    ControlItem(id="MAP-3", function="Map", category="能力与限制",
                description="AI系统的能力边界、已知限制和不确定性已文档化"),
    ControlItem(id="MAP-4", function="Map", category="影响评估",
                description="AI系统对个人、群体和社会的潜在影响已评估"),
    ControlItem(id="MAP-5", function="Map", category="第三方风险评估",
                description="第三方数据、模型和组件的来源及风险已评估"),
    # Measure（MEASURE-1~4）
    ControlItem(id="MEASURE-1", function="Measure", category="评估方法选择",
                description="已选择并验证适当的AI风险评估方法（自动测试/人工评估/红队测试）"),
    ControlItem(id="MEASURE-2", function="Measure", category="可信特征评估",
                description="已评估AI系统的准确性、安全性、公平性、隐私性、可解释性等特征"),
    ControlItem(id="MEASURE-3", function="Measure", category="指标追踪",
                description="已建立AI系统性能和风险的持续追踪指标体系"),
    ControlItem(id="MEASURE-4", function="Measure", category="反馈机制",
                description="已建立评估结果的反馈收集和改进机制"),
    # Manage（MANAGE-1~4）
    ControlItem(id="MANAGE-1", function="Manage", category="风险优先级",
                description="已根据风险严重程度和发生概率对风险进行优先级排序"),
    ControlItem(id="MANAGE-2", function="Manage", category="资源分配",
                description="已分配足够的资源处理已识别的高优先级风险"),
    ControlItem(id="MANAGE-3", function="Manage", category="第三方风险处理",
                description="已制定第三方AI组件的风险处理策略（合同/审计/监控）"),
    ControlItem(id="MANAGE-4", function="Manage", category="风险响应",
                description="已建立风险响应流程（消除/降低/转移/接受）和事件应急方案"),
]

print(f'已定义 {len(NIST_CONTROL_ITEMS)} 个NIST AI RMF控制项')
for func in ['Govern', 'Map', 'Measure', 'Manage']:
    items = [c for c in NIST_CONTROL_ITEMS if c.function == func]
    print(f'  {func}: {len(items)}项')

## TODO2：AI 用例注册表

构建AI用例注册表，包含基于 **OECD AI Incidents Monitor**（https://oecd.ai/en/incidents-overview）真实事件类型设计的AI用例。

每个用例包含两类属性：
- **EU AI Act分类属性**：用于判断是否属于禁止/高风险/有限风险实践
- **NIST评估属性**：用于评估合规得分（has_human_oversight / has_audit_log / has_bias_testing / has_transparency）

**你的任务**：定义 `AIUseCase` 模型，创建至少8个真实AI用例（覆盖营销/HR/金融/医疗/安全等领域）。

In [ ]:
class AIUseCase(BaseModel):
    name: str
    domain: str  # marketing, hr, finance, healthcare, security, etc.
    description: str
    # EU AI Act Article 5: Prohibited practices
    is_subliminal_manipulation: bool = False
    exploits_vulnerabilities: bool = False
    is_social_scoring: bool = False
    predicts_criminal_behavior: bool = False
    untargeted_facial_recognition: bool = False
    emotion_recognition_workplace: bool = False
    biometric_sensitive_attributes: bool = False
    real_time_biometric_id: bool = False
    # EU AI Act Annex III: High-risk AI systems
    is_employment_screening: bool = False
    is_education_assessment: bool = False
    is_credit_scoring: bool = False
    is_insurance_pricing: bool = False
    is_medical_diagnosis: bool = False
    is_judicial_evidence: bool = False
    is_immigration: bool = False
    is_public_service: bool = False
    # EU AI Act Article 50: Limited risk (transparency)
    is_chatbot: bool = False
    generates_content: bool = False
    is_deepfake: bool = False
    # NIST assessment attributes
    has_human_oversight: bool = False
    has_audit_log: bool = False
    has_bias_testing: bool = False
    has_transparency: bool = False

# 真实AI用例（基于OECD AI Incidents Monitor事件类型构建）
AI_USE_CASES = [
    AIUseCase(
        name="AI个性化推荐系统",
        domain="marketing",
        description="电商平台基于用户行为数据的个性化商品推荐系统",
        generates_content=True,
        has_human_oversight=False,
        has_audit_log=True,
        has_bias_testing=False,
        has_transparency=True,
    ),
    AIUseCase(
        name="AI自动文案生成",
        domain="marketing",
        description="使用LLM自动生成多平台营销文案（微信/小红书/抖音）",
        generates_content=True,
        has_human_oversight=True,
        has_audit_log=False,
        has_bias_testing=False,
        has_transparency=False,
    ),
    AIUseCase(
        name="AI动态定价系统",
        domain="marketing",
        description="基于用户数据和市场需求动态调整商品价格",
        has_human_oversight=False,
        has_audit_log=True,
        has_bias_testing=False,
        has_transparency=False,
    ),
    AIUseCase(
        name="AI客服聊天机器人",
        domain="marketing",
        description="面向客户的AI聊天机器人，自动回复客户咨询",
        is_chatbot=True,
        has_human_oversight=True,
        has_audit_log=True,
        has_bias_testing=False,
        has_transparency=True,
    ),
    AIUseCase(
        name="AI简历筛选系统",
        domain="hr",
        description="使用AI自动筛选求职简历，按匹配度排序候选人",
        is_employment_screening=True,
        has_human_oversight=True,
        has_audit_log=True,
        has_bias_testing=False,
        has_transparency=True,
    ),
    AIUseCase(
        name="AI信用评分系统",
        domain="finance",
        description="基于用户信用历史和消费行为评估信用风险",
        is_credit_scoring=True,
        has_human_oversight=True,
        has_audit_log=True,
        has_bias_testing=True,
        has_transparency=True,
    ),
    AIUseCase(
        name="AI人脸识别门禁系统",
        domain="security",
        description="办公区域使用实时人脸识别进行门禁管理",
        real_time_biometric_id=True,
        has_human_oversight=True,
        has_audit_log=True,
        has_bias_testing=False,
        has_transparency=True,
    ),
    AIUseCase(
        name="AI医疗影像诊断",
        domain="healthcare",
        description="使用AI分析医疗影像辅助医生诊断疾病",
        is_medical_diagnosis=True,
        has_human_oversight=True,
        has_audit_log=True,
        has_bias_testing=True,
        has_transparency=True,
    ),
]

print(f'已注册 {len(AI_USE_CASES)} 个AI用例')
for uc in AI_USE_CASES:
    print(f'  [{uc.domain}] {uc.name}')

## TODO3：NIST AI RMF 合规扫描器

实现合规扫描器：对每个AI用例，逐一评估18个NIST控制项的合规分数（0-100）。

评分逻辑（基于用例的治理属性）：

| 控制功能 | 评分因子 |
|---------|---------|
| Govern | 基础20分 + 审计日志 + 透明度 + 人工监督 + 偏见测试 |
| Map | 基础15分 + 透明度 + 审计日志 + 偏见测试 + 人工监督 |
| Measure | 基础10分 + 偏见测试(40分) + 审计日志 + 人工监督 + 透明度 |
| Manage | 基础15分 + 人工监督(35分) + 审计日志 + 透明度 |

**你的任务**：实现 `assess_control()`、`score_to_status()`、`scan_nist_rmf()` 三个函数。

In [ ]:
def assess_control(use_case: AIUseCase, ctrl: ControlItem) -> float:
    """评估单个控制项的合规分数 (0-100)"""
    score = 0.0
    fn = ctrl.function
    if fn == "Govern":
        score += 20  # 基础：政策存在
        if use_case.has_audit_log: score += 20
        if use_case.has_transparency: score += 20
        if use_case.has_human_oversight: score += 20
        if use_case.has_bias_testing: score += 20
    elif fn == "Map":
        score += 15  # 基础：上下文
        if use_case.has_transparency: score += 25
        if use_case.has_audit_log: score += 20
        if use_case.has_bias_testing: score += 20
        if use_case.has_human_oversight: score += 20
    elif fn == "Measure":
        score += 10  # 基础
        if use_case.has_bias_testing: score += 40
        if use_case.has_audit_log: score += 25
        if use_case.has_human_oversight: score += 15
        if use_case.has_transparency: score += 10
    elif fn == "Manage":
        score += 15  # 基础
        if use_case.has_human_oversight: score += 35
        if use_case.has_audit_log: score += 30
        if use_case.has_transparency: score += 20
    return max(0.0, min(100.0, score))

def score_to_status(score: float) -> ComplianceStatus:
    """将分数转为合规状态"""
    if score >= 80: return ComplianceStatus.MET
    elif score >= 50: return ComplianceStatus.PARTIALLY_MET
    elif score >= 1: return ComplianceStatus.NOT_MET
    else: return ComplianceStatus.NOT_ASSESSED

def scan_nist_rmf(use_case: AIUseCase) -> list:
    """对AI用例执行NIST AI RMF合规扫描"""
    results = []
    for ctrl in NIST_CONTROL_ITEMS:
        score = assess_control(use_case, ctrl)
        status = score_to_status(score)
        results.append(ctrl.model_copy(update={"score": score, "status": status}))
    return results

# 验证：扫描第一个用例
test_results = scan_nist_rmf(AI_USE_CASES[0])
print(f'用例: {AI_USE_CASES[0].name}')
for func in ['Govern', 'Map', 'Measure', 'Manage']:
    func_items = [r for r in test_results if r.function == func]
    avg = sum(r.score for r in func_items) / len(func_items)
    print(f'  {func}: 平均分={avg:.1f} ({len(func_items)}项)')

## TODO4：EU AI Act 风险分级器

按EU AI Act真实条款实现风险分级。判定顺序（从严到松）：

1. **Article 5（禁止）**：潜意识操纵、利用脆弱性、社会评分、个体犯罪预测、无差别面部识别、工作场所情感识别、生物特征敏感属性推断、实时远程生物特征识别
2. **Annex III（高风险）**：招聘筛选、教育评估、信贷评估、保险定价、医疗诊断、司法证据、移民管理、公共服务
3. **Article 50（有限风险）**：聊天机器人、AI生成内容、深度伪造 -> 透明度义务
4. **最小风险**：以上均不满足

**你的任务**：实现 `classify_eu_ai_act()` 函数，返回 (风险等级, 判定依据)。

In [ ]:
def classify_eu_ai_act(use_case: AIUseCase):
    """按EU AI Act真实条款判定AI用例的风险等级
    
    Returns:
        tuple: (风险等级, 判定依据)
        风险等级: "禁止" / "高风险" / "有限风险" / "最小风险"
    """
    # --- Article 5: 禁止的AI实践 ---
    prohibited_checks = [
        (use_case.is_subliminal_manipulation, "Article 5: 潜意识操纵"),
        (use_case.exploits_vulnerabilities, "Article 5: 利用特定群体脆弱性"),
        (use_case.is_social_scoring, "Article 5: 社会评分"),
        (use_case.predicts_criminal_behavior, "Article 5: 个体犯罪预测"),
        (use_case.untargeted_facial_recognition, "Article 5: 无差别面部识别数据库抓取"),
        (use_case.emotion_recognition_workplace, "Article 5: 工作场所/教育机构情感识别"),
        (use_case.biometric_sensitive_attributes, "Article 5: 生物特征敏感属性推断"),
        (use_case.real_time_biometric_id, "Article 5: 公共场所实时远程生物特征识别"),
    ]
    for flag, reason in prohibited_checks:
        if flag:
            return ("禁止", reason)
    
    # --- Annex III: 高风险AI系统 ---
    high_risk_checks = [
        (use_case.is_employment_screening, "Annex III: 招聘和人员筛选"),
        (use_case.is_education_assessment, "Annex III: 教育和职业培训评估"),
        (use_case.is_credit_scoring, "Annex III: 信贷评估和信用评分"),
        (use_case.is_insurance_pricing, "Annex III: 保险定价和风险评估"),
        (use_case.is_medical_diagnosis, "Annex III: 医疗诊断和分诊"),
        (use_case.is_judicial_evidence, "Annex III: 司法程序证据评估"),
        (use_case.is_immigration, "Annex III: 移民和边境管理"),
        (use_case.is_public_service, "Annex III: 公共服务资格评估"),
    ]
    for flag, reason in high_risk_checks:
        if flag:
            return ("高风险", reason)
    
    # --- Article 50: 有限风险（透明度义务）---
    if use_case.is_deepfake:
        return ("有限风险", "Article 50: 深度伪造内容需标注")
    if use_case.is_chatbot:
        return ("有限风险", "Article 50: 聊天机器人需告知用户与AI交互")
    if use_case.generates_content:
        return ("有限风险", "Article 50: AI生成内容需以可检测方式标注")
    
    # --- 最小风险 ---
    return ("最小风险", "无额外合规要求，建议参照NIST AI RMF自愿性管理")

# 验证：对所有用例执行EU AI Act分级
print('=== EU AI Act 风险分级结果 ===')
for uc in AI_USE_CASES:
    level, reason = classify_eu_ai_act(uc)
    print(f'  [{uc.domain}] {uc.name} -> {level} ({reason})')

## TODO5：风险热力图分析（pandas）

用 pandas 将合规扫描结果转为DataFrame，按用例×功能（Govern/Map/Measure/Manage）透视，生成风险热力图。

**你的任务**：实现 `build_risk_heatmap()` 函数，返回 (明细DataFrame, 透视DataFrame)。

In [ ]:
def build_risk_heatmap(use_cases, scanner_func):
    """构建NIST AI RMF合规风险热力图
    
    Returns:
        tuple: (明细DataFrame, 透视DataFrame)
    """
    rows = []
    for uc in use_cases:
        results = scanner_func(uc)
        for ctrl in results:
            rows.append({
                "use_case": uc.name,
                "domain": uc.domain,
                "function": ctrl.function,
                "control_id": ctrl.id,
                "category": ctrl.category,
                "score": ctrl.score,
                "status": ctrl.status.value,
            })
    df_detail = pd.DataFrame(rows)
    # 透视：用例 x 功能，取平均分
    df_pivot = df_detail.pivot_table(
        values="score",
        index="use_case",
        columns="function",
        aggfunc="mean"
    )
    # 添加总体平均分列
    df_pivot["总均分"] = df_pivot.mean(axis=1)
    return df_detail, df_pivot

# 执行分析
df_detail, df_pivot = build_risk_heatmap(AI_USE_CASES, scan_nist_rmf)

print('=== 风险热力图（用例 x 功能，平均合规分数）===')
print(df_pivot.to_string())
print(f'\n明细记录数: {len(df_detail)}')
print(f'整体最高分: {df_pivot["总均分"].max():.1f} ({df_pivot["总均分"].idxmax()})')
print(f'整体最低分: {df_pivot["总均分"].min():.1f} ({df_pivot["总均分"].idxmin()})')

## TODO6：营销AI治理专项分析

筛选营销领域的AI用例，结合NIST合规得分和EU AI Act分级，输出营销AI治理分析报告。

分析维度：
- EU AI Act风险等级分布
- NIST合规得分（总体+最弱控制项）
- 治理改进建议

**你的任务**：实现 `marketing_governance_analysis()` 函数，返回分析DataFrame。

In [ ]:
def marketing_governance_analysis(use_cases, classify_func, scanner_func):
    """营销AI治理专项分析
    
    Returns:
        pd.DataFrame: 营销AI用例的治理分析结果
    """
    marketing_cases = [uc for uc in use_cases if uc.domain == "marketing"]
    results = []
    for uc in marketing_cases:
        risk_level, reason = classify_func(uc)
        ctrl_results = scanner_func(uc)
        avg_score = sum(c.score for c in ctrl_results) / len(ctrl_results)
        weakest = min(ctrl_results, key=lambda c: c.score)
        results.append({
            "use_case": uc.name,
            "eu_risk_level": risk_level,
            "eu_reason": reason,
            "nist_avg_score": round(avg_score, 1),
            "weakest_control": f"{weakest.id} ({weakest.function})",
            "weakest_score": weakest.score,
            "weakest_desc": weakest.description,
        })
    return pd.DataFrame(results)

# 执行营销AI治理分析
df_marketing = marketing_governance_analysis(AI_USE_CASES, classify_eu_ai_act, scan_nist_rmf)

print('=== 营销AI治理专项分析 ===')
print(df_marketing.to_string(index=False))

# EU AI Act 分级分布（全部用例）
print('\n=== EU AI Act 风险分级分布（全部用例）===')
all_levels = [classify_eu_ai_act(uc)[0] for uc in AI_USE_CASES]
dist = pd.Series(all_levels).value_counts()
for level, count in dist.items():
    pct = count / len(AI_USE_CASES) * 100
    print(f'  {level}: {count}个 ({pct:.1f}%)')

# 营销AI风险分级分布
print('\n=== 营销AI风险分级分布 ===')
mkt_levels = [classify_eu_ai_act(uc)[0] for uc in AI_USE_CASES if uc.domain == "marketing"]
mkt_dist = pd.Series(mkt_levels).value_counts()
for level, count in mkt_dist.items():
    pct = count / len(mkt_levels) * 100
    print(f'  {level}: {count}个 ({pct:.1f}%)')

# 治理改进建议
print('\n=== 治理改进建议 ===')
for _, row in df_marketing.iterrows():
    gap = 80 - row['nist_avg_score']
    if gap > 0:
        print(f"  [{row['use_case']}] 合规差距={gap:.1f}分，最弱项={row['weakest_control']}({row['weakest_score']}分)")
        print(f"    -> 建议: {row['weakest_desc']}")

## 总结

本练习实现了两个核心AI治理工具：

1. **NIST AI RMF 合规扫描器**：用pydantic定义18个真实控制项，对AI用例按Govern/Map/Measure/Manage四大功能评分
2. **EU AI Act 风险分级器**：按Article 5（禁止）-> Annex III（高风险）-> Article 50（有限风险）-> 最小风险的真实条款逻辑分级

**关键发现**：
- 营销AI系统（推荐/文案/客服）多属于"有限风险"（需透明度标注），但NIST合规得分差异大
- 缺少偏见测试（bias testing）的AI系统在Measure维度得分最低，是最常见合规短板
- 没有人工监督的自动化系统在Manage维度得分低，需要增加human-in-the-loop

**下一步**：在Day 2中，你将学习如何用LangGraph编排Agent工作流，并把今天的治理控制点嵌入到工作流中。